<div dir="rtl">
  <p style="color: #80f9fa; text-align: right;">
    <b>معماری کانولوشن در شبکه‌های عصبی</b>
  </p>
</div>

---

<div dir="rtl" style="text-align: right;">
دستورات کانولوشن دوبُعدی:
</div>

```python
torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)
```

- in_channels (int) – Number of channels in the input image

- out_channels (int) – Number of channels produced by the convolution

- kernel_size (int or tuple) – Size of the convolving kernel

- stride (int or tuple, optional) – Stride of the convolution. Default: 1

- padding (int, tuple or str, optional) – Padding added to all four sides of the input. Default: 0

- dilation (int or tuple, optional) – Spacing between kernel elements. Default: 1

- groups (int, optional) – Number of blocked connections from input channels to output channels. Default: 1

- bias (bool, optional) – If True, adds a learnable bias to the output. Default: True

- padding_mode (str, optional) – 'zeros', 'reflect', 'replicate' or 'circular'. Default: 'zeros'

```python
torch.nn.MaxPool2d(kernel_size, stride=None, padding=0, dilation=1, return_indices=False, ceil_mode=False)
```

- kernel_size (Union[int, tuple[int, int]]) – the size of the window to take a max over

- stride (Union[int, tuple[int, int]]) – the stride of the window. Default value is kernel_size

- padding (Union[int, tuple[int, int]]) – Implicit negative infinity padding to be added on both sides

- dilation (Union[int, tuple[int, int]]) – a parameter that controls the stride of elements in the window

- return_indices (bool) – if True, will return the max indices along with the outputs.

- ceil_mode (bool) – when True, will use ceil instead of floor to compute the output shape

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc = nn.Linear(32 * 8 * 8, 10)
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device)
dummy_input = torch.randn(1, 3, 32, 32).to(device)
output = model(dummy_input)

In [55]:
import torch

class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = torch.nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, stride=1, dilation=1)
        self.relu = torch.nn.ReLU()
        self.conv2 = torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1, stride=2)
        self.pool = torch.nn.MaxPool2d(kernel_size=2)
        self.flatten = torch.nn.Flatten()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        return x


x = torch.randn(1, 3, 32, 32)
model = Model()
y = model(x)
print(y.shape)

torch.Size([1, 2048])


<div dir="rtl" style="text-align: right;">
تمرین: معماری فوق را به نحوی تصحیح نمایید که از
nn.Sequential
به جای تعریف چندین متغیر استفاده شود.
</div>

<div dir="rtl" style="text-align: right;">
سوال:
عملکرد هر یک از لایه‌های زیر را بررسی نموده و مقایسه نمایید.
فرض: ورودی سه کاناله و دارای ابعاد ۳۲ در ۳۲ است.

- کانولوشن با اندازه کرنل ۳، تعداد کانال خروجی ۳۲، پدینگ ۱ و استراید ۱
- کانولوشن با اندازه کرنل ۳، تعداد کانال خروجی ۳۲، پدینگ ۱ و استراید ۲
- کانولوشن با اندازه کرنل ۳، تعداد کانال خروجی ۳۲، پدینگ ۱، استراید ۱ و مکس‌پولینگ ۲

</div>

<div dir="rtl" style="text-align: right;">
مقداردهی اولیه وزن‌ها
</div>

In [ ]:
import torch
import torch.nn as nn

class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = torch.nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, stride=1, dilation=1)

        nn.init.xavier_normal_(self.conv1.weight)
        nn.init.xavier_uniform_(self.conv1.weight)
        nn.init.kaiming_normal_(self.conv1.weight)
        nn.init.ones_(self.conv1.weight)    # test
        nn.init.zeros_(self.conv1.weight)   # bias

    def forward(self, x):
        x = self.conv1(x)
        return x


x = torch.randn(1, 3, 32, 32)
model = Model()
y = model(x)
print(y.shape)
for name, param in model.named_parameters():
    print(name, param.shape)

torch.Size([1, 16, 32, 32])
conv1.weight torch.Size([16, 3, 3, 3])
conv1.bias torch.Size([16])


<div dir="rtl" style="text-align: right;">
کانولوشن depthwise
</div>

In [ ]:
import torch

# Depthwise Separable
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = torch.nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, padding=1, stride=1, dilation=1, groups=3)
        self.pointwise = torch.nn.Conv2d(in_channels=3, out_channels=15, kernel_size=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pointwise(x)
        return x


x = torch.randn(1, 3, 32, 32)
model = Model()
y = model(x)
print(y.shape)

torch.Size([1, 15, 32, 32])


<div dir="rtl" style="text-align: right;">
batch normalization
</div>

```python
torch.nn.BatchNorm2d(num_features, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, device=None, dtype=None)
```

- num_features (int) – C from an expected input of size (N,C,H,W)

- eps (float) – a value added to the denominator for numerical stability. Default: 1e-5

- momentum (Optional[float]) – the value used for the running_mean and running_var computation. Can be set to None for cumulative moving average (i.e. simple average). Default: 0.1

- affine (bool) – a boolean value that when set to True, this module has learnable affine parameters. Default: True

- track_running_stats (bool) – a boolean value that when set to True, this module tracks the running mean and variance, and when set to False, this module does not track such statistics, and initializes statistics buffers running_mean and running_var as None. When these buffers are None, this module always uses batch statistics. in both training and eval modes. Default: True

In [43]:
m = nn.BatchNorm2d(100)
for n, p in m.named_parameters():
    print(n, p.shape)
inputs = torch.randn(20, 100, 35, 45)
outputs = m(inputs)
print(torch.mean(inputs), torch.mean(outputs))
torch.std(inputs), torch.std(outputs)

weight torch.Size([100])
bias torch.Size([100])
tensor(-0.0002) tensor(-1.3951e-09, grad_fn=<MeanBackward0>)


(tensor(1.0001), tensor(1.0000, grad_fn=<StdBackward0>))

---

<div dir="rtl" style="text-align: right;">
تمرین:
یک شبکه عصبی کانولوشنی برای داده‌ی
mnist
براساس معماری زیر و مطابق با قالب پیوست، آموزش بدهید.

معماری پیشنهادی
- (Sequential Block):
    - Conv2D(32, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(32, 3×3, padding=1) → ReLU → MaxPool(2×2)

- (Sequential Block):
    - Conv2D(64, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(64, 3×3, padding=1) → ReLU → MaxPool(2×2)

- لایه‌های Fully Connected (Dense) در مدل Sequential:
    - Flatten()
    - Dense(512) → ReLU → Dropout(0.4)
    - Dense(256) → ReLU → Dropout(0.4)
    - Dense(128) → ReLU
    - Dense(10)
</div>

---

<div dir="rtl" style="text-align: right;">
تمرین:
کانولوشن‌های به‌کاررفته در سوال قبل را با نسخه‌ی
depthwise
و
pointwise
معادلش جایگزین نموده و نتیجه را مقایسه نمایید.
</div>

---

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self, num_classes=10):
        super(Model, self).__init__()
        

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding="same"),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )


        self.conv_block2 = nn.Sequential(
            # nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding="same"),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding="same", groups=32),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=1, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )


        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.fc_block(x)
        return x

x = torch.randn(1, 1, 28, 28)
model = Model()
y = model(x)
print(y.shape)

torch.Size([1, 10])


<div dir="rtl" style="text-align: right;">
بلوک residual:
</div>

In [ ]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(in_channels)

    def forward(self, x):
        residual = x  # Skip connection
        out = self.conv1(x)
        out = self.bn1(out)
        out = torch.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual  # Adding the input (skip connection)
        out = torch.relu(out)
        return out

x = torch.randn(1, 3, 32, 32)
model = ResidualBlock(3)
y = model(x)
print(y.shape)